# Comparacao RAG Estruturado (v2)

Esta versao usa:
- **GraphRAG** (mesmo fluxo do projeto)
- **Naive RAG com OpenSearch** (modulo independente `naive-rag-opensearch`)

Inclui indexacao dos documentos no OpenSearch dentro do notebook.

A **avaliacao de recuperacao** usa as metricas **Context Precision** e **Context Recall** do [RAGAS](https://docs.ragas.io/) (juizo por LLM sobre a utilidade dos contextos e sobre a cobertura da resposta de referencia). No GraphRAG, os contextos incluem entidades, relacionamentos, claims/covariates e trechos (`context_docs` empacotado pelo fluxo local/global), nao apenas os TextUnits como no RAG tradicional (chunks do OpenSearch). Variaveis opcionais: `RAGAS_LLM_MODEL` (default: mesmo modelo do GraphRAG) e `RAGAS_MAX_CONTEXT_BLOCKS` (default 16; limite de blocos por pergunta para custo da metrica de precisao).

In [3]:
from __future__ import annotations

import csv
import importlib
import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "graphrag-neo4j-langchain").is_dir():
    if (PROJECT_ROOT.parent / "graphrag-neo4j-langchain").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent

GRAPHRAG_SRC = PROJECT_ROOT / "graphrag-neo4j-langchain" / "src"
NAIVE_OS_SRC = PROJECT_ROOT / "naive-rag-opensearch"
DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
QA_CSV = (PROJECT_ROOT / "docs" / "rag_eval_qa_estruturado.csv").resolve()

if not GRAPHRAG_SRC.is_dir():
    raise FileNotFoundError(f"Nao encontrei graphrag em: {GRAPHRAG_SRC}")
if not NAIVE_OS_SRC.is_dir():
    raise FileNotFoundError(f"Nao encontrei modulo naive-rag-opensearch em: {NAIVE_OS_SRC}")

for p in (str(GRAPHRAG_SRC), str(NAIVE_OS_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DOCS_DIR: {DOCS_DIR}")
print(f"QA_CSV: {QA_CSV}")

PROJECT_ROOT: /home/micael/mestrado/benchmarking-graphrag
DOCS_DIR: /home/micael/mestrado/benchmarking-graphrag/pdfs_txt
QA_CSV: /home/micael/mestrado/benchmarking-graphrag/docs/rag_eval_qa_estruturado.csv


In [4]:
from dotenv import load_dotenv
import socket

# Carrega .env do GraphRAG e do modulo Naive RAG OpenSearch.
load_dotenv(PROJECT_ROOT / "graphrag-neo4j-langchain" / ".env", override=False)
load_dotenv(PROJECT_ROOT / "naive-rag-opensearch" / ".env", override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY nao definida. Configure em um dos .env.")

# Ajusta host/porta do OpenSearch para funcionar tanto dentro quanto fora do
# container 'comparison'.
os_host = os.getenv("OPENSEARCH_HOST", "localhost")
os_port = os.getenv("OPENSEARCH_PORT", "9200")

# Dentro da rede docker-compose: API HTTP do OpenSearch em opensearch:9200.
if os_host == "opensearch" and os_port == "9300":
    os.environ["OPENSEARCH_PORT"] = "9200"
    os_port = "9200"

# Fora do compose, o hostname 'opensearch' nao resolve; use porta publicada.
if os_host == "opensearch":
    try:
        socket.getaddrinfo("opensearch", int(os_port))
    except OSError:
        os.environ["OPENSEARCH_HOST"] = "localhost"
        os_host = "localhost"
        if os_port == "9200":
            os.environ["OPENSEARCH_PORT"] = "9300"
            os_port = "9300"

print("OPENAI_API_KEY: OK")
print("OPENSEARCH_HOST:", os_host)
print("OPENSEARCH_PORT:", os_port)
print("OPENSEARCH_INDEX:", os.getenv("OPENSEARCH_INDEX", "naive_rag_chunks"))

OPENAI_API_KEY: OK
OPENSEARCH_HOST: localhost
OPENSEARCH_PORT: 9300
OPENSEARCH_INDEX: naive_rag_chunks


In [5]:
@dataclass
class QARow:
    qid: str
    tipo_pergunta: str
    question: str
    trecho_1: str
    fonte_trecho_1: str
    trecho_2: str
    fonte_trecho_2: str
    reference_answer: str


def load_qa_estruturado_csv(path: Path) -> list[QARow]:
    required = {
        "id",
        "tipo_pergunta",
        "pergunta",
        "trecho_1",
        "fonte_trecho_1",
        "trecho_2",
        "fonte_trecho_2",
        "resposta_esperada",
    }
    rows: list[QARow] = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        miss = required - set(reader.fieldnames or [])
        if miss:
            raise ValueError(f"CSV incompleto. Faltam colunas: {sorted(miss)}")
        for r in reader:
            rows.append(
                QARow(
                    qid=(r.get("id") or "").strip(),
                    tipo_pergunta=(r.get("tipo_pergunta") or "").strip(),
                    question=(r.get("pergunta") or "").strip(),
                    trecho_1=(r.get("trecho_1") or "").strip(),
                    fonte_trecho_1=(r.get("fonte_trecho_1") or "").strip(),
                    trecho_2=(r.get("trecho_2") or "").strip(),
                    fonte_trecho_2=(r.get("fonte_trecho_2") or "").strip(),
                    reference_answer=(r.get("resposta_esperada") or "").strip(),
                )
            )
    if not rows:
        raise ValueError("Nenhuma linha valida no CSV")
    return rows


qa_rows = load_qa_estruturado_csv(QA_CSV)
print(f"Perguntas carregadas: {len(qa_rows)}")

Perguntas carregadas: 12


## 1) Indexacao no OpenSearch (Naive RAG)

Antes de comparar, execute esta secao para criar/atualizar o indice vetorial no OpenSearch.

In [6]:
from naive_rag_opensearch.rag import index_documents, retrieve, answer_question
from opensearchpy.exceptions import ConnectionError as OpenSearchConnectionError

try:
    indexed = index_documents(DOCS_DIR)
except OpenSearchConnectionError as exc:
    raise RuntimeError(
        "Falha ao conectar no OpenSearch. Verifique se o container esta no ar e se "
        "OPENSEARCH_HOST/OPENSEARCH_PORT correspondem ao ambiente atual "
        "(dentro do compose: opensearch:9200; fora: localhost:9300)."
    ) from exc

print(f"Chunks indexados no OpenSearch: {indexed}")

# Smoke test de busca
preview = retrieve("Qual e a data do incidente?", k=3)
print(f"Top-3 recuperados: {len(preview)}")
for i, d in enumerate(preview, start=1):
    print(f"[{i}] {d.get('source')} | {d.get('chunk_id')} | score={d.get('score')}")

Chunks indexados no OpenSearch: 39
Top-3 recuperados: 3
[1] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-0 | score=0.47348982
[2] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-1 | score=0.47194734
[3] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-2 | score=0.467656


## 2) Funcoes GraphRAG e Naive RAG (OpenSearch)

In [7]:
def run_graphrag(question: str, thread_id: str) -> dict:
    import graphrag.graph.nodes as graph_nodes
    import graphrag.graph.query_graph as query_graph

    # Recarrega para refletir alteracoes recentes sem reiniciar kernel.
    importlib.reload(graph_nodes)
    importlib.reload(query_graph)

    compiled = query_graph.get_compiled_graph()
    cfg = {"configurable": {"thread_id": f"nb-v2-{thread_id}"}}
    try:
        return compiled.invoke({"question": question}, config=cfg)
    except Exception as exc:
        return {
            "final_answer": "",
            "context_docs": [],
            "cypher_result": "",
            "cypher_error": f"{type(exc).__name__}: {exc}",
        }


def graphrag_retrieved_context_strings(state: dict) -> list[str]:
    """Cada elemento e um bloco do contexto entregue ao LLM (entidades, relacionamentos, trechos, etc.)."""
    out: list[str] = []
    for doc in state.get("context_docs") or []:
        if not isinstance(doc, dict):
            continue
        txt = (doc.get("page_content") or "").strip()
        if txt:
            out.append(txt)
    return out


def naive_opensearch_top_k(question: str, k: int) -> tuple[list[str], list[str], str]:
    # Garante funcionamento mesmo se a celula de indexacao/import nao tiver sido executada no kernel atual.
    _retrieve = globals().get("retrieve")
    _answer_question = globals().get("answer_question")

    if _retrieve is None or _answer_question is None:
        try:
            from naive_rag_opensearch.rag import retrieve as _retrieve, answer_question as _answer_question
        except Exception as exc:
            raise RuntimeError(
                "Funcoes do Naive RAG OpenSearch indisponiveis. "
                "Execute a celula de indexacao/import (secao 1) antes do benchmark."
            ) from exc

    hits = _retrieve(question, k=k)
    ids = [str(h.get("chunk_id") or "") for h in hits]
    texts = [str(h.get("text") or "") for h in hits]

    rag_out = _answer_question(question, k=k)
    answer = (rag_out.get("answer") or "").strip()
    return ids, texts, answer

## 3) Executar comparacao (Naive OpenSearch vs GraphRAG)

In [8]:
import nest_asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, ContextRecall

from graphrag.config import LLM_MODEL

_ragas_model = os.getenv("RAGAS_LLM_MODEL", LLM_MODEL)
_ragas_embeddings_provider = os.getenv("RAGAS_EMBEDDINGS_PROVIDER", "openai")
_ragas_embeddings_model = os.getenv("RAGAS_EMBEDDINGS_MODEL", "text-embedding-3-small")
_ragas_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
_ragas_llm = llm_factory(_ragas_model, client=_ragas_client)

# Compatibilidade entre versoes do RAGAS:
# - algumas esperam embedding_factory(provider, model=...)
# - outras podem aceitar apenas provider
try:
    _ragas_embeddings = embedding_factory(
        _ragas_embeddings_provider,
        model=_ragas_embeddings_model,
        client=_ragas_client,
    )
except TypeError:
    _ragas_embeddings = embedding_factory(_ragas_embeddings_provider)

_metric_context_precision = ContextPrecision(llm=_ragas_llm)
_metric_context_recall = ContextRecall(llm=_ragas_llm)
_metric_answer_relevancy = AnswerRelevancy(llm=_ragas_llm, embeddings=_ragas_embeddings)

print(f"RAGAS: modelo de juizo = {_ragas_model}")
print(f"RAGAS: provider de embeddings = {_ragas_embeddings_provider}")
print(f"RAGAS: modelo de embeddings = {_ragas_embeddings_model}")

_ragas_max_ctx = int(os.getenv("RAGAS_MAX_CONTEXT_BLOCKS", "16"))
print(f"RAGAS: max blocos de contexto por pergunta = {_ragas_max_ctx}")


def _as_float_metric(x) -> float:
    try:
        v = float(x)
    except (TypeError, ValueError):
        return float("nan")
    return v if not math.isnan(v) else float("nan")


async def ragas_context_recall_precision(
    question: str, reference: str, retrieved_contexts: list[str]
) -> tuple[float, float]:
    """Retorna (context_recall, context_precision) do RAGAS; NaN se faltar referencia ou contexto."""
    ref = (reference or "").strip()
    ctxs = [c.strip() for c in retrieved_contexts if (c or "").strip()][:_ragas_max_ctx]
    if not ref or not ctxs:
        return float("nan"), float("nan")
    try:
        cr = await _metric_context_recall.ascore(
            user_input=question, reference=ref, retrieved_contexts=ctxs
        )
        cp = await _metric_context_precision.ascore(
            user_input=question, reference=ref, retrieved_contexts=ctxs
        )
        return _as_float_metric(cr.value), _as_float_metric(cp.value)
    except Exception as exc:
        print(f"RAGAS (aviso): {type(exc).__name__}: {exc}")
        return float("nan"), float("nan")


async def ragas_answer_relevancy(question: str, answer: str) -> float:
    """Retorna a metrica Answer Relevancy do RAGAS; NaN se faltar resposta."""
    ans = (answer or "").strip()
    if not ans:
        return float("nan")
    try:
        ar = await _metric_answer_relevancy.ascore(user_input=question, response=ans)
        return _as_float_metric(ar.value)
    except Exception as exc:
        print(f"RAGAS (aviso): {type(exc).__name__}: {exc}")
        return float("nan")


/home/micael/mestrado/benchmarking-graphrag/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_15639/2022309989.py:22: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  _ragas_embeddings = embedding_factory(


RAGAS: modelo de juizo = gpt-4.1-2025-04-14
RAGAS: provider de embeddings = openai
RAGAS: modelo de embeddings = text-embedding-3-small
RAGAS: max blocos de contexto por pergunta = 16


In [9]:
K = 5


async def _run_benchmark_comparison() -> list[dict]:
    async def _ragas_eval(question: str, reference: str, retrieved_contexts: list[str]) -> tuple[float, float]:
        """Versao local para evitar usar definicao antiga no kernel."""
        ref = (reference or "").strip()
        ctxs = [c.strip() for c in retrieved_contexts if (c or "").strip()][:_ragas_max_ctx]
        if not ref or not ctxs:
            return float("nan"), float("nan")
        try:
            cr = await _metric_context_recall.ascore(
                user_input=question, reference=ref, retrieved_contexts=ctxs
            )
            cp = await _metric_context_precision.ascore(
                user_input=question, reference=ref, retrieved_contexts=ctxs
            )
            return _as_float_metric(cr.value), _as_float_metric(cp.value)
        except Exception as exc:
            print(f"RAGAS (aviso): {type(exc).__name__}: {exc}")
            return float("nan"), float("nan")

    results: list[dict] = []
    for i, row in enumerate(qa_rows):
        tag = str(abs(hash(row.question)) % 10_000_000)

        n_ids, n_texts, n_ans = naive_opensearch_top_k(row.question, K)
        n_cr, n_cp = await _ragas_eval(row.question, row.reference_answer, n_texts)
        n_ar = await ragas_answer_relevancy(row.question, n_ans)

        g_state = run_graphrag(row.question, tag)
        g_ctxs = graphrag_retrieved_context_strings(g_state if isinstance(g_state, dict) else {})
        print('contexto para avaliação do graphrag: ', g_ctxs)
        g_ans = (g_state.get("final_answer") or "").strip() if isinstance(g_state, dict) else ""
        g_cr, g_cp = await _ragas_eval(row.question, row.reference_answer, g_ctxs)
        g_ar = await ragas_answer_relevancy(row.question, g_ans)

        g_error = (g_state.get("cypher_error") or "").strip() if isinstance(g_state, dict) else ""
        if g_error:
            print(f"[{i + 1}/{len(qa_rows)}] GraphRAG fallback: {g_error[:160]}")

        results.append(
            {
                "qid": row.qid,
                "q": i + 1,
                "hop": row.tipo_pergunta,
                "system": "naive_rag_opensearch",
                "context_recall": n_cr,
                "context_precision": n_cp,
                "answer_relevancy": n_ar,
                "question": row.question,
                "reference": row.reference_answer,
                "generated": n_ans,
                "retrieved_ids": " | ".join(n_ids),
            }
        )
        results.append(
            {
                "qid": row.qid,
                "q": i + 1,
                "hop": row.tipo_pergunta,
                "system": "graphrag",
                "context_recall": g_cr,
                "context_precision": g_cp,
                "answer_relevancy": g_ar,
                "question": row.question,
                "reference": row.reference_answer,
                "generated": g_ans,
                "retrieved_ids": "",
            }
        )

        n_cr_show = n_cr if not math.isnan(n_cr) else -1.0
        g_cr_show = g_cr if not math.isnan(g_cr) else -1.0
        print(f"[{i + 1}/{len(qa_rows)}] {row.tipo_pergunta} naive_cr={n_cr_show:.3f} graphrag_cr={g_cr_show:.3f}")

    print("Concluido.")
    return results


results = await _run_benchmark_comparison()

Decision: local
Subqueries: [SubQuery(sub_query='Qual foi a data da explosão na Accurate Energetic Systems?'), SubQuery(sub_query='Encontre registros de incidentes envolvendo explosões na Accurate Energetic Systems e suas datas.')]


/home/micael/mestrado/benchmarking-graphrag/graphrag-neo4j-langchain/src/graphrag/store/neo4j_graph.py:14: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  _graph = Neo4jGraph(


Local retrieve: 8 seed entities, 30 context chunks
Graph QA Cypher (attempt 1): MATCH (e:Entity {name: "Deadly Blast at Tennessee Plant"})-[:HAS_CLAIM]->(c:Claim)
WHERE c.text CONTAINS "October 2025"
RETURN c.text
LIMIT 25
contexto para avaliação do graphrag:  ['=== Afirmação (claim) [Inspection: 1855764.015] ===\nInspection 1855764.015 was conducted by the Occupational Safety and Health Administration for Accurate Energetic Systems, LLC.', '=== Afirmação (claim) [AES] ===\nThe incident resulted in approximately $4.3 million in property damage.', '=== Afirmação (claim) [AES] ===\nAES had the capacity to pour from Kettle B and Kettle A concurrently to produce two different product lines.', '=== Entidades relevantes (âncoras) ===\n- AES (Organization): Accurate Energetic Systems, LLC (AES), manufacturer of explosives\n---\nCompany operating the facility and melt/pour process.\n---\nOperator of Building 602 and the complex\n- AES facility (Facility): Facility operated by Accurate Energeti

In [10]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    df = pd.DataFrame(results)
    display_cols = ["q", "qid", "hop", "system", "context_recall", "context_precision", "answer_relevancy"]
    display(df[display_cols])

    summary = (
        df.groupby("system")[["context_recall", "context_precision", "answer_relevancy"]]
        .mean(numeric_only=True)
        .rename_axis("media macro")
    )
    display(summary)
else:
    from statistics import mean

    for sys_name in sorted(set(r["system"] for r in results)):
        sub = [r for r in results if r["system"] == sys_name]

        def mkey(k: str):
            vals = [x[k] for x in sub if isinstance(x[k], (int, float)) and not math.isnan(x[k])]
            return mean(vals) if vals else float("nan")

        print(
            sys_name,
            "context_recall",
            mkey("context_recall"),
            "context_precision",
            mkey("context_precision"),
            "answer_relevancy",
            mkey("answer_relevancy"),
        )

,q,qid,hop,system,context_recall,context_precision,answer_relevancy
0,1,Q01,single-hop,naive_rag_opensearch,1.0,0.866667,0.992382
1,1,Q01,single-hop,graphrag,1.0,0.215909,0.929250
2,2,Q02,single-hop,naive_rag_opensearch,1.0,0.833333,1.000000
3,2,Q02,single-hop,graphrag,1.0,0.141560,0.711254
4,3,Q03,single-hop,naive_rag_opensearch,1.0,0.804167,1.000000
5,3,Q03,single-hop,graphrag,1.0,1.000000,1.000000
6,4,Q04,multi-hop,naive_rag_opensearch,1.0,1.000000,1.000000
7,4,Q04,multi-hop,graphrag,1.0,0.066667,0.969634
8,5,Q05,multi-hop,naive_rag_opensearch,1.0,1.000000,0.616046
9,5,Q05,multi-hop,graphrag,1.0,0.700000,0.870064


,context_recall,context_precision,answer_relevancy
media macro,,,
graphrag,0.909091,0.365746,0.850632
naive_rag_opensearch,1.000000,0.864236,0.754218


In [11]:
try:
    import pandas as pd
except ImportError:
    pd = None


def _normaliza_categoria(hop: str) -> str:
    h = (hop or "").strip().lower().replace("_", "-")
    if "single" in h:
        return "single-hop"
    if "multi" in h:
        return "multi-hop"
    if "global" in h:
        return "global"
    if "tabela" in h:
        return "tabela"
    return h


if pd is not None:
    df = pd.DataFrame(results).copy()
    df["categoria"] = df["hop"].map(_normaliza_categoria)

    categorias_alvo = ["single-hop", "multi-hop", "global", "tabela"]
    abordagens_alvo = ["graphrag", "naive_rag_opensearch"]

    tabela_media_answer_relevancy = (
        df[df["categoria"].isin(categorias_alvo) & df["system"].isin(abordagens_alvo)]
        .groupby(["categoria", "system"], as_index=False)["answer_relevancy"]
        .mean(numeric_only=True)
    )

    tabela_media_answer_relevancy = (
        tabela_media_answer_relevancy.pivot(index="categoria", columns="system", values="answer_relevancy")
        .reindex(categorias_alvo)
        .reindex(columns=abordagens_alvo)
    )

    display(tabela_media_answer_relevancy)
else:
    from statistics import mean

    categorias_alvo = ["single-hop", "multi-hop", "global", "tabela"]
    abordagens_alvo = ["graphrag", "naive_rag_opensearch"]

    agrupado = {(c, s): [] for c in categorias_alvo for s in abordagens_alvo}
    for r in results:
        c = _normaliza_categoria(r.get("hop", ""))
        s = r.get("system", "")
        v = r.get("answer_relevancy")
        if c in categorias_alvo and s in abordagens_alvo and isinstance(v, (int, float)) and not math.isnan(v):
            agrupado[(c, s)].append(v)

    for c in categorias_alvo:
        print(f"\nCategoria: {c}")
        for s in abordagens_alvo:
            vals = agrupado[(c, s)]
            m = mean(vals) if vals else float("nan")
            print(f"  {s}: {m}")

system,graphrag,naive_rag_opensearch
categoria,,
single-hop,0.880168,0.997461
multi-hop,0.831090,0.783077
global,0.808647,0.288977
tabela,0.882624,0.947357
